# Week 2 Day 1 — Agent Foundations: Reasoning Loops, Tool Calling & Raw Python Agents

**Model:** `gemini-2.5-flash` via the `google-genai` SDK
**Constraint:** No LangChain / LangGraph — everything below is raw Python.

This notebook builds a minimal ReAct-style agent from scratch:
1. Task 1 — Agent concepts & mental model
2. Task 2 — Tool calling fundamentals (single-turn)
3. Task 3 — Minimal agent loop (multi-turn, multi-tool)
4. Task 4 — Memory & state handling + logging
5. Task 5 — Failure modes & guardrails


## Task 1: Agent Concepts & Mental Model

**Chatbot** — single-turn (or multi-turn) text response. It reasons in words but never *acts* on the world; it can't run a calculation, fetch live data, or read a file on its own. Input in, text out.

**Workflow** — a fixed, pre-programmed sequence of steps (e.g. "call API A, then B, then format output"). The order of operations is decided by the *developer* ahead of time, not by the model at run time.

**Agent** — a model that decides *at run time*, step by step, which action to take next (including whether to call a tool), observes the result, and re-plans. Control flow is decided by the model, not hardcoded by the developer.

**What makes something "agentic"?**
- **Autonomy** — it chooses its own next step instead of following a fixed script.
- **Tool use** — it can act on the world (calculators, APIs, file I/O) rather than only producing text.
- **Multi-step planning** — it can decompose a goal into a sequence of sub-actions.
- **Self-correction** — it can notice a tool failed or an answer was wrong, and try a different approach instead of stopping.

**The ReAct pattern (Reason → Act → Observe → repeat)**

```
User goal
   │
   ▼
┌─────────┐      ┌────────┐      ┌───────────┐
│ REASON  │ ───▶ │  ACT   │ ───▶ │  OBSERVE  │
│ (think, │      │ (call a│      │ (get tool │
│  decide)│      │  tool) │      │  result)  │
└─────────┘      └────────┘      └───────────┘
     ▲                                  │
     └──────────── repeat ──────────────┘
                (until the model
                 has enough info to
                 give a final answer)
```

Pseudocode:
```
loop:
    response = model(history, tools)
    if response has function_call:
        result = execute_tool(response.function_call)
        history.append(function_call, function_result)
    else:
        return response.text   # done
```

**When is an agent overkill?**
If the task is a single deterministic transformation — e.g. "translate this sentence" or "sum these two numbers with a known formula" — a plain prompt or a short script is faster, cheaper, and far more predictable than a tool-calling loop. Agents earn their cost only when the *number and order of steps* isn't known in advance.


## Setup

API key is read from the `GEMINI_API_KEY` environment variable — never hardcoded.


In [12]:
import os
import json
import time
import logging

from google import genai
from google.genai import types

# ---- Reproducibility -------------------------------------------------
SEED = 42

# ---- Logging (this is the debugging habit referenced in Task 4) ------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("agent")

# ---- API key: environment variable only, never hardcoded -------------
API_KEY = os.getenv("GEMINI_API_KEY")
if not API_KEY:
    log.warning("GEMINI_API_KEY not set. Set it before running live calls, e.g.:")
    log.warning("  export GEMINI_API_KEY=your_key_here")

client = genai.Client(api_key=API_KEY) if API_KEY else None
MODEL_NAME = "gemini-3.5-flash"


## Task 2: Tool Calling Fundamentals

Two tools, each with a proper JSON schema (name, description, parameters):

1. **`calculator`** — performs basic arithmetic.
2. **`get_weather`** — mock weather lookup for a city (stub, no real network call).

**Why descriptions matter:** the model never sees our Python code — it only sees the tool's `name`, `description`, and `parameters` schema. A vague description ("does math") makes the model unsure *when* to call the tool and *what* arguments to pass. A precise description ("Evaluates a basic arithmetic expression with +, -, *, /, and parentheses") lets the model reliably decide "this is the right tool" and construct valid, correctly-typed arguments.


In [13]:
# --- Tool implementations (the actual Python functions we execute) ----

def calculator(operation: str, a: float, b: float):
    """Performs a basic arithmetic operation on two numbers."""
    try:
        if operation == "add":
            return a + b
        elif operation == "subtract":
            return a - b
        elif operation == "multiply":
            return a * b
        elif operation == "divide":
            if b == 0:
                return {"error": "division by zero"}
            return a / b
        else:
            return {"error": f"unsupported operation: {operation}"}
    except Exception as e:
        return {"error": str(e)}


MOCK_WEATHER_DB = {
    "london": {"condition": "cloudy", "temp_c": 15},
    "paris": {"condition": "sunny", "temp_c": 21},
    "lahore": {"condition": "hot, hazy", "temp_c": 38},
    "new york": {"condition": "rainy", "temp_c": 18},
}

def get_weather(location: str):
    """Returns mock current weather data for a given city."""
    key = location.strip().lower()
    if key in MOCK_WEATHER_DB:
        return {"location": location, **MOCK_WEATHER_DB[key]}
    return {"error": f"no weather data for '{location}'"}


# --- JSON schemas (function declarations) sent to the Gemini API ------

calculator_schema = {
    "name": "calculator",
    "description": (
        "Evaluates a basic arithmetic operation (add, subtract, multiply, "
        "divide) between two numbers. Use this whenever the user asks for "
        "a numeric computation instead of estimating the answer yourself."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "operation": {
                "type": "string",
                "enum": ["add", "subtract", "multiply", "divide"],
                "description": "The arithmetic operation to perform.",
            },
            "a": {"type": "number", "description": "The first operand."},
            "b": {"type": "number", "description": "The second operand."},
        },
        "required": ["operation", "a", "b"],
    },
}

weather_schema = {
    "name": "get_weather",
    "description": (
        "Gets the current mock weather (condition and temperature in Celsius) "
        "for a given city. Use this whenever the user asks about weather in "
        "a specific location."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "location": {
                "type": "string",
                "description": "The city name, e.g. 'London' or 'Paris'.",
            }
        },
        "required": ["location"],
    },
}

TOOLS = [calculator_schema, weather_schema]
TOOL_FUNCTIONS = {"calculator": calculator, "get_weather": get_weather}

gemini_tools = [types.Tool(function_declarations=TOOLS)]
print(json.dumps(TOOLS, indent=2))


[
  {
    "name": "calculator",
    "description": "Evaluates a basic arithmetic operation (add, subtract, multiply, divide) between two numbers. Use this whenever the user asks for a numeric computation instead of estimating the answer yourself.",
    "parameters": {
      "type": "object",
      "properties": {
        "operation": {
          "type": "string",
          "enum": [
            "add",
            "subtract",
            "multiply",
            "divide"
          ],
          "description": "The arithmetic operation to perform."
        },
        "a": {
          "type": "number",
          "description": "The first operand."
        },
        "b": {
          "type": "number",
          "description": "The second operand."
        }
      },
      "required": [
        "operation",
        "a",
        "b"
      ]
    }
  },
  {
    "name": "get_weather",
    "description": "Gets the current mock weather (condition and temperature in Celsius) for a given city. Use th

### Single-turn tool call demo

One request → model chooses a tool → we manually execute it → we send the result back for a final answer. This mirrors the `function_call` / `function_result` round trip described in the assignment brief, expressed with the actual `google-genai` SDK.


In [14]:
def single_turn_tool_demo(prompt: str):
    """Demonstrates one manual function_call -> function_result round trip."""
    if client is None:
        log.error("No client configured (missing GEMINI_API_KEY). Skipping live call.")
        return None

    contents = [types.Content(role="user", parts=[types.Part(text=prompt)])]

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=contents,
        config=types.GenerateContentConfig(tools=gemini_tools),
    )

    candidate_part = response.candidates[0].content.parts[0]

    if candidate_part.function_call is None:
        log.info("Model answered directly (no tool call): %s", response.text)
        return response.text

    fc = candidate_part.function_call
    log.info("Model requested tool call: %s(%s)", fc.name, dict(fc.args))

    tool_fn = TOOL_FUNCTIONS[fc.name]
    result = tool_fn(**fc.args)
    log.info("Tool result: %s", result)

    contents.append(response.candidates[0].content)
    contents.append(
        types.Content(
            role="user",
            parts=[
                types.Part.from_function_response(
                    name=fc.name,
                    response={"result": result},
                )
            ],
        )
    )

    final_response = client.models.generate_content(
        model=MODEL_NAME,
        contents=contents,
        config=types.GenerateContentConfig(tools=gemini_tools),
    )
    log.info("Final answer: %s", final_response.text)
    return final_response.text


# Example (requires GEMINI_API_KEY to actually run):
single_turn_tool_demo("What's the weather in London?")


17:52:29 | INFO | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-flash:generateContent "HTTP/1.1 200 OK"
17:52:29 | INFO | Model requested tool call: get_weather({'location': 'London'})
17:52:29 | INFO | Tool result: {'location': 'London', 'condition': 'cloudy', 'temp_c': 15}
17:52:30 | INFO | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-flash:generateContent "HTTP/1.1 200 OK"
17:52:30 | INFO | Final answer: The weather in London is currently cloudy with a temperature of 15°C.


'The weather in London is currently cloudy with a temperature of 15°C.'

## Task 3: Minimal Agent Loop

A `while`-loop agent:
1. Send message (+ history) to Gemini with tools.
2. If the response contains a `function_call` → execute it → append the `function_result` → loop again.
3. If not → return the final text answer.

`max_iterations` guards against infinite loops (Task 5's first failure mode).


In [15]:
class SimpleAgent:
    def __init__(self, client, model_name, tools_schema, tool_functions, max_iterations=6):
        self.client = client
        self.model_name = model_name
        self.tools_schema = tools_schema
        self.tool_functions = tool_functions
        self.max_iterations = max_iterations
        self.gemini_tools = [types.Tool(function_declarations=tools_schema)]

    def run(self, user_prompt: str):
        if self.client is None:
            log.error("No client configured (missing GEMINI_API_KEY). Cannot run live agent.")
            return None

        history = [types.Content(role="user", parts=[types.Part(text=user_prompt)])]
        log.info("=== New task: %s ===", user_prompt)

        for step in range(1, self.max_iterations + 1):
            log.info("--- Iteration %d/%d ---", step, self.max_iterations)

            response = self.client.models.generate_content(
                model=self.model_name,
                contents=history,
                config=types.GenerateContentConfig(tools=self.gemini_tools),
            )

            part = response.candidates[0].content.parts[0]
            history.append(response.candidates[0].content)

            if part.function_call is None:
                log.info("REASON: model has enough information. Returning final answer.")
                return response.text

            fc = part.function_call
            log.info("REASON -> ACT: calling %s(%s)", fc.name, dict(fc.args))

            if fc.name not in self.tool_functions:
                # Failure mode: hallucinated / undefined tool name
                observation = {"error": f"tool '{fc.name}' is not defined"}
                log.warning("Hallucinated tool call: %s", fc.name)
            else:
                try:
                    observation = self.tool_functions[fc.name](**fc.args)
                except Exception as e:
                    # Failure mode: tool raised an exception / bad args
                    observation = {"error": str(e)}
                    log.warning("Tool execution error: %s", e)

            log.info("OBSERVE: %s", observation)

            history.append(
                types.Content(
                    role="user",
                    parts=[
                        types.Part.from_function_response(
                            name=fc.name,
                            response={"result": observation},
                        )
                    ],
                )
            )

        log.warning("Max iterations (%d) reached without a final answer.", self.max_iterations)
        return "[agent stopped: max_iterations reached]"


agent = SimpleAgent(client, MODEL_NAME, TOOLS, TOOL_FUNCTIONS, max_iterations=6)

# Multi-step test requiring 2+ tool calls:
answer = agent.run("Look up the weather in London and Paris and tell me which is warmer.")
print("\nFINAL ANSWER:", answer)


17:52:30 | INFO | === New task: Look up the weather in London and Paris and tell me which is warmer. ===
17:52:30 | INFO | --- Iteration 1/6 ---
17:52:31 | INFO | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-flash:generateContent "HTTP/1.1 200 OK"
17:52:31 | INFO | REASON -> ACT: calling get_weather({'location': 'London'})
17:52:31 | INFO | OBSERVE: {'location': 'London', 'condition': 'cloudy', 'temp_c': 15}
17:52:31 | INFO | --- Iteration 2/6 ---
17:52:32 | INFO | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-flash:generateContent "HTTP/1.1 200 OK"
17:52:32 | INFO | REASON: model has enough information. Returning final answer.



FINAL ANSWER: 


## Task 4: Memory & State Handling

- **Conversation memory (message history)** — the full back-and-forth transcript (user turns, model turns, function calls, function results). In the Gemini `interactions` API this is what `previous_interaction_id` chains together; in the `generate_content` SDK path used above, we carry it manually as the growing `history` list passed into every call. It's what lets the model "remember" what was already asked and answered.

- **Working memory (scratchpad / mid-task state)** — information the agent is tracking *during* a single task that isn't necessarily part of the visible conversation: intermediate tool results, a running total, which sub-goals are done. In our `SimpleAgent`, this is the `observation` value computed each loop iteration and fed back in as a `function_response` — the agent's "scratchpad" is really just the accumulating `history`, since we don't keep a separate state object.

The two overlap in this simple design (both live in `history`), but conceptually they're different: conversation memory is "what was said", working memory is "what the agent has figured out so far in service of the current goal". A more advanced agent would keep working memory in a separate structured object (e.g. a dict of sub-task statuses) instead of inline text.

**Logging** — every iteration above already logs the REASON / ACT / OBSERVE steps via the `logging` module (not just `print`), so the full trace is timestamped and level-tagged. This is the debugging habit worth carrying into every future framework (LangChain, LangGraph, CrewAI, etc.).


## Task 5: Failure Modes & Guardrails

We deliberately break the agent in three ways and observe what happens.


In [ ]:
import time

# Scenario 1: Ambiguous request
answer = agent.run("Is it nice out?")   
print(answer)
time.sleep(25)  
# Scenario 2: A tool that returns an error
answer = agent.run("What is 10 divided by 0?")
print(answer)
time.sleep(25)  

# Scenario 3: A task requiring a tool we haven't defined
answer = agent.run("Send an email to my professor saying I will be late.")
print(answer)

17:56:48 | INFO | === New task: Is it nice out? ===
17:56:48 | INFO | --- Iteration 1/6 ---
17:56:50 | INFO | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-flash:generateContent "HTTP/1.1 200 OK"
17:56:50 | INFO | REASON: model has enough information. Returning final answer.


To tell you if it's nice out, could you let me know which city or location you're in?


17:57:15 | INFO | === New task: What is 10 divided by 0? ===
17:57:15 | INFO | --- Iteration 1/6 ---
17:57:17 | INFO | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-flash:generateContent "HTTP/1.1 200 OK"
17:57:17 | INFO | REASON -> ACT: calling calculator({'a': 10, 'b': 0, 'operation': 'divide'})
17:57:17 | INFO | OBSERVE: {'error': 'division by zero'}
17:57:17 | INFO | --- Iteration 2/6 ---
17:57:18 | INFO | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-flash:generateContent "HTTP/1.1 200 OK"
17:57:18 | INFO | REASON: model has enough information. Returning final answer.


Division by zero is mathematically undefined. Therefore, 10 divided by 0 does not have a valid numerical answer.


17:57:43 | INFO | === New task: Send an email to my professor saying I will be late. ===
17:57:43 | INFO | --- Iteration 1/6 ---
17:57:46 | INFO | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-flash:generateContent "HTTP/1.1 200 OK"
17:57:46 | INFO | REASON: model has enough information. Returning final answer.


I don't have the ability to send emails directly, but I can certainly write a draft for you! You can copy and paste the text below into your email client:

***

**Subject:** Late for Class Today - [Your Name] - [Course Name/Number]

Dear Professor [Professor's Last Name],

I am writing to let you know that I will be late for our class today, [Course Name/Number], due to [briefly mention reason, e.g., unexpected traffic / a personal delay]. 

I expect to arrive at approximately [Estimated Arrival Time]. I apologize for the interruption my late arrival may cause and will slip in as quietly as possible. I will also make sure to get any notes or announcements I missed from a classmate.

Thank you for your understanding.

Sincerely,

[Your Name]  
[Your Student ID Number]  
[Your Class Section/Time]


**Observed behavior (from running the above against the live API):**

- **Scenario 1 (ambiguous):** the model either asks a clarifying question in plain text (no tool call) or picks a reasonable default and states its assumption — it doesn't call a tool blindly.
- **Scenario 2 (tool error):** `calculator` returns `{"error": "division by zero"}`; the loop feeds that back as the `function_result`, and the model explains the division is undefined instead of crashing.
- **Scenario 3 (undefined tool):** the model either says it has no way to send email, or (if it hallucinates a call to a nonexistent tool like `send_email`) our `SimpleAgent.run` catches the unknown name and returns `{"error": "tool 'send_email' is not defined"}` as the observation, which the model then reports back to the user instead of pretending it worked.

### Failure modes & mitigations

| Failure Mode | Mitigation |
|---|---|
| Infinite loops | `max_iterations` safeguard (hard cap in `SimpleAgent.run`) |
| Hallucinated tool calls (model invents a tool name) | Validate `fc.name` against `TOOL_FUNCTIONS` before executing; return a structured error observation instead of crashing |
| Wrong / missing tool arguments | Validate arguments against the JSON schema (types, required fields) before calling; wrap the call in `try/except` |
| Silent errors inside a tool | Wrap tool execution in `try/except`, return the error as the `function_result` so the model can react to it |
| Model doesn't call a tool when it should | Strengthen the tool `description` and/or add a system instruction nudging tool use for relevant queries |
| Tool execution timeout / hang | Wrap tool calls with a per-tool timeout (e.g. `signal.alarm` or a thread with a timeout) and return an error observation on expiry |

### Why do frameworks like LangChain / LangGraph / CrewAI exist?

Having built this by hand, the value of these frameworks is obvious: they standardize the plumbing we just wrote manually — history management, tool-schema validation, retries, streaming, memory backends, multi-agent orchestration, and provider-agnostic tool calling — so teams don't reimplement the same `while`-loop, error handling, and logging boilerplate for every project. The trade-off is abstraction: frameworks hide exactly the mechanics (function_call → execute → function_result → repeat) that this exercise was designed to make visible. Understanding the raw loop first makes it much easier to debug what a framework is doing under the hood when something goes wrong.
